# λ_d Phase 2 — Russian Dataset
Training on 750M tokens (Russian Wikipedia + PD books)

**Auto-resume:** если сессия оборвалась, перезапусти эту ячейку — она
подхватит последний чекпоинт с Drive.

**Drive layout:**
- `MyDrive/lambda/russian_chunks.npy` — данные (3 GB)
- `MyDrive/lambda/model_start.pt` — стартовая модель (229 MB)
- `MyDrive/lambda/lambda_colab.zip` — код
- `MyDrive/lambda/model_step*.pt` — чекпоинты (только 3 последних)
- `MyDrive/lambda/model_best.pt` — лучший чекпоинт

In [ ]:
# ─── 1. Install & Mount ────────────────────────────────────────────────
import os, sys, subprocess, time, math, shutil, glob, re, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Install dependencies (if needed)
subprocess.run(['pip', 'install', '-q', 'numpy', 'torch'], capture_output=True)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
DRIVE = '/content/drive/MyDrive/lambda'
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('code', exist_ok=True)

# Copy code from Drive
shutil.unpack_archive(os.path.join(DRIVE, 'lambda_colab.zip'), 'code')
sys.path.insert(0, 'code')
sys.path.insert(0, 'code/ld_model')

# Copy data (3 GB, ~5 min)
DATA_PATH = 'russian_chunks.npy'
if not os.path.exists(DATA_PATH):
    print('Copying data from Drive...')
    shutil.copy2(os.path.join(DRIVE, 'russian_chunks.npy'), DATA_PATH)

print('Ready. Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ─── 2. Config + mmap Dataset ────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
D = 896
VOCAB = 50000
N_MODES = 4
N_LAYERS = 12
SEQ_LEN = 128
LR = 1e-3
WARMUP_FRAC = 0.05
EPOCHS = 3
GRAD_CLIP = 1.0
LOG_EVERY = 200
CKPT_EVERY = 5000
KEEP_CKPTS = 3

# Load data via memmap — ~0 MB RAM, reads from disk on demand
arr = np.load(DATA_PATH, mmap_mode='r')
n_total = arr.shape[0]
n_eval = min(500, n_total // 20)
n_train = n_total - n_eval
print(f'Total chunks: {n_total}, train: {n_train}, eval: {n_eval}')

class ChunkDataset(torch.utils.data.Dataset):
    """Reads one chunk at a time from memmap — ~4 KB/batch RAM."""
    def __init__(self, arr, start, end):
        self.arr = arr
        self.start = start
        self.n = end - start
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        i = self.start + idx
        x = torch.from_numpy(self.arr[i, :-1].copy()).to(torch.long)
        y = torch.from_numpy(self.arr[i, 1:].copy()).to(torch.long)
        return x, y

train_ds = ChunkDataset(arr, 0, n_train)
eval_ds  = ChunkDataset(arr, n_train, n_train + n_eval)

In [ ]:
# ─── 3. Model ───────────────────────────────────────────────────────────
from ld_model.core import LDConfig, LDStack

class Phase2Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D)
        cfg = LDConfig()
        cfg.D = D; cfg.n_layers = N_LAYERS; cfg.n_modes = N_MODES
        cfg.vocab = VOCAB; cfg.bottleneck = 256
        self.stack = LDStack(cfg)
        self.lm_head = nn.Linear(D, VOCAB, bias=False)
    def forward(self, x):
        return self.lm_head(self.stack(self.embed(x)))

model = Phase2Model().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

In [ ]:
# ─── 4. Load / Resume ───────────────────────────────────────────────────
def find_latest(path):
    best, best_n = None, -1
    if not os.path.exists(path):
        return None
    for f in os.listdir(path):
        # model_step*.pt + model_interrupt_step*.pt + model_best.pt
        m = re.match(r'model(?:_interrupt)?_step(\d+)\.pt', f)
        if m:
            n = int(m.group(1))
            if n > best_n:
                best_n, best = n, os.path.join(path, f)
        # Fallback: model_best.pt если нет step-файлов
        if f == 'model_best.pt' and best is None:
            best = os.path.join(path, f)
    return best

NEW_BS = 64  # current batch_size

def load_ckpt(path):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
    if 'model_fp16' in ckpt:
        sd = {k: v.float() if v.dtype == torch.float16 else v
              for k, v in ckpt['model_fp16'].items()}
    elif 'model_state_dict' in ckpt:
        sd = ckpt['model_state_dict']
    else:
        sd = ckpt  # грузим сам state_dict
    model.load_state_dict(sd, strict=False)
    old_bs = ckpt.get('batch_size', 64)  # auto-detect
    step = int(ckpt.get('step', 0) * old_bs / NEW_BS)  # rescale
    epoch = ckpt.get('epoch', 0)
    ppl = ckpt.get('best_ppl', float('inf'))
    return step, epoch, ppl

step, epoch, best_ppl = 0, 0, float('inf')
ckpt = find_latest(DRIVE) or find_latest('checkpoints')
if ckpt:
    print(f'Resuming from {ckpt}...')
    shutil.copy2(ckpt, 'checkpoints/' + os.path.basename(ckpt))
    step, epoch, best_ppl = load_ckpt(ckpt)
    print(f'  step={step}, epoch={epoch}, best_ppl={best_ppl:.1f}')
else:
    # Start from phase2_best.pt (wikitext pretrain) — reset counters
    print('Starting from wikitext pretrain...')
    start_path = os.path.join(DRIVE, 'model_start.pt')
    if os.path.exists(start_path):
        load_ckpt(start_path)  # load weights, ignore counters
        step, epoch, best_ppl = 0, 0, float('inf')
        print(f'  Loaded model_start.pt, reset step=0 epoch=0')

def clean_old():
    files = []
    if not os.path.exists('checkpoints'):
        return
    for f in os.listdir('checkpoints'):
        m = re.match(r'model(?:_interrupt)?_step(\d+)\.pt', f)
        if m:
            files.append((int(m.group(1)), os.path.join('checkpoints', f)))
    files.sort(reverse=True)
    for _, p in files[KEEP_CKPTS:]:
        os.remove(p)

In [ ]:
# ─── 5. Train Loop ─────────────────────────────────────────────────────
batch_size = NEW_BS
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=0)
eval_loader  = DataLoader(eval_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=0)

total_steps = (n_train // batch_size) * EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
print(f'Total steps: {total_steps}, warmup: {warmup_steps}')

def get_lr(s):
    if s < warmup_steps:
        return LR * (s + 1) / warmup_steps
    p = (s - warmup_steps) / max(total_steps - warmup_steps, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * p))

def save_to_drive(path):
    shutil.copy2(os.path.join('checkpoints', path), os.path.join(DRIVE, path))
    clean_old()

t_start = time.time()
try:
    for ep in range(epoch, EPOCHS):
        epoch = ep  # track for interrupt handler
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE, non_blocking=True), by.to(DEVICE, non_blocking=True)
            loss = F.cross_entropy(model(bx).reshape(-1, VOCAB), by.reshape(-1))
            if torch.isnan(loss):
                print(f'  [NAN] step {step}, skip')
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            lr = get_lr(step)
            for g in optimizer.param_groups:
                g['lr'] = lr
            optimizer.step()
            optimizer.zero_grad()
            epoch_loss += loss.item()
            step += 1
            n_batches += 1

            if step % LOG_EVERY == 0:
                ppl = math.exp(epoch_loss / n_batches)
                print(f'  Step {step:5d} | loss={epoch_loss/n_batches:.4f} | ppl={ppl:.1f} | lr={lr:.2e}')

            if step % CKPT_EVERY == 0:
                fname = f'model_step{step}.pt'
                sd = {k: v.half() if v.dtype==torch.float32 else v for k,v in model.state_dict().items()}
                torch.save({'model_fp16': sd, 'step': step, 'epoch': ep, 'best_ppl': best_ppl, 'batch_size': NEW_BS},
                           os.path.join('checkpoints', fname))
                save_to_drive(fname)
                clean_old()

        train_ppl = math.exp(epoch_loss / n_batches) if n_batches else float('inf')
        model.eval()
        eval_loss = 0.0
        with torch.no_grad():
            for bx, by in eval_loader:
                bx, by = bx.to(DEVICE, non_blocking=True), by.to(DEVICE, non_blocking=True)
                eval_loss += F.cross_entropy(model(bx).reshape(-1, VOCAB), by.reshape(-1)).item()
        eval_ppl = math.exp(eval_loss / len(eval_loader))
        print(f'>> Epoch {ep+1}: train_ppl={train_ppl:.1f}, eval_ppl={eval_ppl:.1f}')

        if eval_ppl < best_ppl:
            best_ppl = eval_ppl
            sd = {k: v.half() if v.dtype==torch.float32 else v for k,v in model.state_dict().items()}
            torch.save({'model_fp16': sd, 'step': step, 'epoch': ep+1, 'best_ppl': best_ppl, 'batch_size': NEW_BS},
                       os.path.join('checkpoints', 'model_best.pt'))
            save_to_drive('model_best.pt')

        fname = f'model_epoch{ep+1}.pt'
        sd = {k: v.half() if v.dtype==torch.float32 else v for k,v in model.state_dict().items()}
        torch.save({'model_fp16': sd, 'step': step, 'epoch': ep+1, 'best_ppl': best_ppl, 'batch_size': NEW_BS},
                   os.path.join('checkpoints', fname))
        save_to_drive(fname)

    print(f'\nDone. Time: {time.time()-t_start:.0f}s')
    print(f'Best eval PPL: {best_ppl:.1f}')

except KeyboardInterrupt:
    print('\n[SIGINT] Saving...')
    fname = f'model_interrupt_step{step}.pt'
    sd = {k: v.half() if v.dtype==torch.float32 else v for k,v in model.state_dict().items()}
    torch.save({'model_fp16': sd, 'step': step, 'epoch': epoch, 'best_ppl': best_ppl, 'batch_size': NEW_BS},
               os.path.join('checkpoints', fname))
    save_to_drive(fname)